# 실습 6: 특성 엔지니어링 + 학습 데이터 준비

추출된 숫자형 특성을 스케일링하고, 학습/테스트 데이터를 분할합니다.
10주차 모델 학습에 바로 사용할 수 있는 데이터를 저장합니다.

**실행 방법**: JupyterLab에서 셀을 위에서부터 `Shift+Enter`로 순차 실행

**사전 준비**: `preprocessing.ipynb`를 먼저 실행해 `csic2010_cleaned.csv` 생성

**다음 단계**: `approach_preview.ipynb`

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os

print("=" * 65)
print("  [실습 6] 특성 엔지니어링 + 학습 데이터 준비")
print("=" * 65)

# 전처리된 데이터 로드
# Jupyter / 스크립트 양쪽에서 동작하도록 SCRIPT_DIR을 결정
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

data_path = os.path.join(SCRIPT_DIR, "csic2010_cleaned.csv")
if not os.path.exists(data_path):
    print(f"\n  !! csic2010_cleaned.csv 파일이 없습니다.")
    print(f"  >> 먼저 preprocessing.py를 실행하세요.")
    raise SystemExit('필수 파일이 없어 중단합니다')

df = pd.read_csv(data_path)
print(f"\n  데이터 로드: {df.shape[0]:,}행 x {df.shape[1]}열")

  [실습 6] 특성 엔지니어링 + 학습 데이터 준비

  데이터 로드: 10,000행 x 44열


## Step 1: 라벨 인코딩

In [3]:
print(f"\n{'─' * 65}")
print("  Step 1: 라벨 인코딩")
print(f"{'─' * 65}")

# 이진 분류 라벨 (Normal=0, Anomalous=1)
df["is_attack"] = (df["label"] == "Anomalous").astype(int)

normal_count = (df["is_attack"] == 0).sum()
attack_count = (df["is_attack"] == 1).sum()

print(f"\n  정상 (Normal=0):    {normal_count:>8,}건 ({normal_count/len(df)*100:.1f}%)")
print(f"  공격 (Anomalous=1): {attack_count:>8,}건 ({attack_count/len(df)*100:.1f}%)")


─────────────────────────────────────────────────────────────────
  Step 1: 라벨 인코딩
─────────────────────────────────────────────────────────────────

  정상 (Normal=0):       5,895건 (59.0%)
  공격 (Anomalous=1):    4,105건 (41.0%)


## Step 2: 특성(Feature) 선택

In [4]:
print(f"\n{'─' * 65}")
print("  Step 2: 특성 선택")
print(f"{'─' * 65}")

# preprocessing.py에서 추출한 숫자형 특성
feature_cols = [
    # 길이 관련
    "url_length", "body_length", "total_length",
    # URL 구조
    "num_params", "path_depth", "has_query", "query_length", "body_num_params",
    # 공격 키워드
    "has_sql", "sql_keyword_count", "has_xss",
    "has_traversal", "traversal_count",
    "has_cmd_injection", "has_crlf",
    # 특수문자 / 공백
    "num_special_chars", "special_char_ratio",
    "num_dots", "num_slashes", "num_spaces",
    # 통계
    "url_entropy", "digit_ratio", "upper_ratio",
]

# 존재하지 않는 컬럼 제외
available_cols = [c for c in feature_cols if c in df.columns]
missing_cols = [c for c in feature_cols if c not in df.columns]

if missing_cols:
    print(f"\n  !! 누락된 특성: {missing_cols}")
    print(f"  >> preprocessing.py를 다시 실행하세요.")

feature_cols = available_cols

print(f"\n  사용할 특성: {len(feature_cols)}개")
print(f"  특성 목록:")
for i, col in enumerate(feature_cols, 1):
    mean_val = df[col].mean()
    print(f"    {i:2d}. {col:<25s}  (평균: {mean_val:.3f})")


─────────────────────────────────────────────────────────────────
  Step 2: 특성 선택
─────────────────────────────────────────────────────────────────

  사용할 특성: 23개
  특성 목록:
     1. url_length                 (평균: 59.301)
     2. body_length                (평균: 31.226)
     3. total_length               (평균: 90.527)
     4. num_params                 (평균: 1.636)
     5. path_depth                 (평균: 3.015)
     6. has_query                  (평균: 0.290)
     7. query_length               (평균: 30.087)
     8. body_num_params            (평균: 1.705)
     9. has_sql                    (평균: 0.116)
    10. sql_keyword_count          (평균: 0.123)
    11. has_xss                    (평균: 0.013)
    12. has_traversal              (평균: 0.000)
    13. traversal_count            (평균: 0.000)
    14. has_cmd_injection          (평균: 0.009)
    15. has_crlf                   (평균: 0.016)
    16. num_special_chars          (평균: 3.123)
    17. special_char_ratio         (평균: 0.019)
    18. num_dots        

## Step 3: 결측치 처리 (숫자형 특성)

In [5]:
print(f"\n{'─' * 65}")
print("  Step 3: 결측치 처리")
print(f"{'─' * 65}")

missing = df[feature_cols].isnull().sum()
missing_total = missing.sum()
print(f"\n  결측치 총 개수: {missing_total}")

if missing_total > 0:
    for col in feature_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"    {col}: 중앙값({median_val:.2f})으로 대체")
    print(f"  처리 후 결측치: {df[feature_cols].isnull().sum().sum()}")
else:
    print("  결측치 없음!")


─────────────────────────────────────────────────────────────────
  Step 3: 결측치 처리
─────────────────────────────────────────────────────────────────

  결측치 총 개수: 0
  결측치 없음!


## Step 4: 학습/테스트 데이터 분할 (80:20)

In [6]:
print(f"\n{'─' * 65}")
print("  Step 4: 학습/테스트 데이터 분할 (80:20)")
print(f"{'─' * 65}")

X = df[feature_cols].values
y = df["is_attack"].values

# stratify: 정상/공격 비율을 학습/테스트에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n  학습 데이터: {X_train.shape[0]:,}건")
print(f"  테스트 데이터: {X_test.shape[0]:,}건")
print(f"  학습 공격 비율: {y_train.mean()*100:.1f}%")
print(f"  테스트 공격 비율: {y_test.mean()*100:.1f}%")
print(f"  >> stratify 덕분에 비율이 동일합니다!")


─────────────────────────────────────────────────────────────────
  Step 4: 학습/테스트 데이터 분할 (80:20)
─────────────────────────────────────────────────────────────────

  학습 데이터: 8,000건
  테스트 데이터: 2,000건
  학습 공격 비율: 41.0%
  테스트 공격 비율: 41.0%
  >> stratify 덕분에 비율이 동일합니다!


## Step 5: 특성 스케일링 (StandardScaler)

In [7]:
print(f"\n{'─' * 65}")
print("  Step 5: 특성 스케일링 (StandardScaler)")
print(f"{'─' * 65}")

scaler = StandardScaler()

# ★ 핵심: 학습 데이터로 fit, 테스트 데이터는 transform만!
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled = scaler.transform(X_test)          # transform만

print(f"\n  스케일링 전 (url_length):")
url_idx = feature_cols.index("url_length")
print(f"    학습 평균: {X_train[:, url_idx].mean():.2f}, "
      f"표준편차: {X_train[:, url_idx].std():.2f}")
print(f"\n  스케일링 후:")
print(f"    학습 평균: {X_train_scaled[:, url_idx].mean():.4f}, "
      f"표준편차: {X_train_scaled[:, url_idx].std():.4f}")

print(f"\n  >> 왜 따로 할까?")
print(f"     테스트 데이터 = '미래의 새로운 HTTP 요청'")
print(f"     미래 데이터의 통계를 미리 알 수 없으므로,")
print(f"     학습 데이터의 통계로만 변환 = 데이터 누수(Data Leakage) 방지!")


─────────────────────────────────────────────────────────────────
  Step 5: 특성 스케일링 (StandardScaler)
─────────────────────────────────────────────────────────────────

  스케일링 전 (url_length):
    학습 평균: 59.36, 표준편차: 71.97

  스케일링 후:
    학습 평균: -0.0000, 표준편차: 1.0000

  >> 왜 따로 할까?
     테스트 데이터 = '미래의 새로운 HTTP 요청'
     미래 데이터의 통계를 미리 알 수 없으므로,
     학습 데이터의 통계로만 변환 = 데이터 누수(Data Leakage) 방지!


In [10]:
X_train_scaled

array([[-0.46356561, -0.43147891, -0.70119357, ..., -0.87548658,
        -0.46047515, -0.77657952],
       [ 0.53687573, -0.43147891,  0.08440095, ...,  1.1940257 ,
        -0.02912402,  0.59165647],
       [ 3.45482964, -0.43147891,  2.37571827, ...,  2.24308886,
         0.80635704,  0.18782759],
       ...,
       [-0.57472576, -0.43147891, -0.78848185, ..., -1.14649732,
        -0.26069147, -0.77657952],
       [-0.64420085, -0.43147891, -0.84303702, ..., -1.63172923,
        -0.01988078, -0.77657952],
       [-0.31072041, -0.43147891, -0.58117218, ..., -0.22879944,
        -0.59782643, -0.77657952]], shape=(8000, 23))

In [11]:
X_train

array([[2.60000000e+01, 0.00000000e+00, 2.60000000e+01, ...,
        3.73814933e+00, 3.70370370e-02, 0.00000000e+00],
       [9.80000000e+01, 0.00000000e+00, 9.80000000e+01, ...,
        4.68336648e+00, 7.07070707e-02, 5.05050505e-02],
       [3.08000000e+02, 0.00000000e+00, 3.08000000e+02, ...,
        5.16250956e+00, 1.35922330e-01, 3.55987055e-02],
       ...,
       [1.80000000e+01, 0.00000000e+00, 1.80000000e+01, ...,
        3.61436945e+00, 5.26315789e-02, 0.00000000e+00],
       [1.30000000e+01, 0.00000000e+00, 1.30000000e+01, ...,
        3.39274741e+00, 7.14285714e-02, 0.00000000e+00],
       [3.70000000e+01, 0.00000000e+00, 3.70000000e+01, ...,
        4.03351350e+00, 2.63157895e-02, 0.00000000e+00]], shape=(8000, 23))

## Step 6: 클래스 불균형 확인

In [8]:
print(f"\n{'─' * 65}")
print("  Step 6: 클래스 불균형 확인")
print(f"{'─' * 65}")

normal_train = (y_train == 0).sum()
attack_train = (y_train == 1).sum()
ratio = normal_train / attack_train

print(f"\n  학습 데이터:")
normal_bar = "█" * int(normal_train / len(y_train) * 40)
attack_bar = "█" * int(attack_train / len(y_train) * 40)
print(f"    정상:  {normal_train:>8,}건 {normal_bar}")
print(f"    공격:  {attack_train:>8,}건 {attack_bar}")
print(f"\n  정상:공격 비율 = {ratio:.1f}:1")
print(f"  >> class_weight='balanced' 옵션으로 10주차에 대응합니다")


─────────────────────────────────────────────────────────────────
  Step 6: 클래스 불균형 확인
─────────────────────────────────────────────────────────────────

  학습 데이터:
    정상:     4,716건 ███████████████████████
    공격:     3,284건 ████████████████

  정상:공격 비율 = 1.4:1
  >> class_weight='balanced' 옵션으로 8주차에 대응합니다


## Step 7: 10주차용 데이터 저장 (pickle)

In [9]:
print(f"\n{'─' * 65}")
print("  Step 7: 10주차용 데이터 저장")
print(f"{'─' * 65}")

# LLM 테스트용 샘플 (원본 텍스트 포함 — LLM에 HTTP 요청을 보여줄 때 사용)
text_cols = ["method", "url", "url_decoded", "body", "body_decoded",
             "full_text", "label"]
available_text_cols = [c for c in text_cols if c in df.columns]
llm_sample = df[available_text_cols + feature_cols + ["is_attack"]].sample(
    500, random_state=42
)

processed_data = {
    "X_train": X_train_scaled,
    "X_test": X_test_scaled,
    "X_train_raw": X_train,
    "X_test_raw": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_cols": feature_cols,
    "scaler": scaler,
    "llm_sample": llm_sample,
}

output_path = os.path.join(SCRIPT_DIR, "processed_data.pkl")
with open(output_path, "wb") as f:
    pickle.dump(processed_data, f)

print(f"\n  저장 완료: processed_data.pkl")
print(f"    학습 데이터: {X_train_scaled.shape}")
print(f"    테스트 데이터: {X_test_scaled.shape}")
print(f"    특성 수: {len(feature_cols)}개")
print(f"    LLM 테스트 샘플: {len(llm_sample)}건 (원본 텍스트 포함)")


─────────────────────────────────────────────────────────────────
  Step 7: 10주차용 데이터 저장
─────────────────────────────────────────────────────────────────

  저장 완료: processed_data.pkl
    학습 데이터: (8000, 23)
    테스트 데이터: (2000, 23)
    특성 수: 23개
    LLM 테스트 샘플: 500건 (원본 텍스트 포함)


## 전체 요약

In [ ]:
print(f"\n{'=' * 65}")
print("  7주차 전체 작업 요약")
print(f"{'=' * 65}")

print("""
  1교시: 웹 보안 이론 + 공격 유형 체험
    -> attack_examples.py, attack_quiz.py

  2교시: CSIC 2010 데이터 로드 + EDA + 시각화
    -> data_load_explore.ipynb, eda_keyword_analysis.ipynb, eda_visualization.ipynb / eda_visualization.py (Streamlit)

  3교시: 전처리 + 특성 엔지니어링 + 학습 데이터 준비
    -> preprocessing.py, feature_engineering.py (지금 이 파일)

  생성된 데이터 파일:
    - csic2010_requests.csv    (파싱된 HTTP 요청 데이터)
    - csic2010_cleaned.csv     (전처리 + 특성 추출 완료)
    - processed_data.pkl       (10주차 모델 학습용)
""")

print("  >> 8주차에서 processed_data.pkl을 로드하여")
print("     통계/ML/LLM 3가지 방법으로 분류 비교 실험을 합니다!")
print(f"{'=' * 65}")